# 06 -- Model Training Boundary

> **Data mode: SYNTHETIC FIXTURE DATA.** Every filing/price/universe/sector record in this notebook comes from `_fixtures.py`, hand-authored for demonstration only. **Nothing computed in this notebook is a genuine historical result, and no cell here may be cited as evidence of reproducing `report_current.html`.** See `docs/reproducibility_findings.md` for the actual reproducibility status.

Demonstrates `train_production_model`'s dependency gate. This notebook does **not** substitute a fake estimator to force a result -- if scikit-learn is unavailable it reports `blocked=True` and stops, exactly like a real production run would.

In [ ]:
import atlas_quant
print("atlas_quant path:", atlas_quant.__file__)

import sys
sys.path.insert(0, ".")  # this notebook's own directory (for _fixtures.py) -- not a legacy path hack
import _fixtures


In [ ]:
from atlas_quant.strategies.filing_momentum_ml.config import FilingMomentumModelConfig
from atlas_quant.strategies.filing_momentum_ml.model_schema import FeatureMatrix
from atlas_quant.strategies.filing_momentum_ml.production.model_boundary import train_production_model
from atlas_quant.strategies.filing_momentum_ml.training_dataset import TrainingDatasetResult, TrainingEligibilityResult
from datetime import date, datetime

dataset = TrainingDatasetResult(
    target_quarter_end=date(2023, 3, 31), training_cutoff=datetime(2023, 5, 12),
    candidate_quarters=(), included_quarters=(), excluded_quarters=(),
    feature_matrix=FeatureMatrix(rows=(), instrument_ids=(), feature_timestamps=(), rejected=()),
    labels=(), instrument_ids=(), feature_timestamps=(), label_available_timestamps=(),
    positive_label_count=0, negative_label_count=0, total_row_count=0, quarter_count=0,
    model_schema_identity="notebook-demo",
)
eligibility = TrainingEligibilityResult(
    eligible=True, quarter_count=8, min_train_quarters=8, quarter_gate_passed=True,
    positive_label_count=10, n_winners=10, positive_label_gate_passed=True,
    non_empty=True, matrix_label_length_match=True, has_finite_values=True,
    both_classes_present=True, reasons=(),
)
result = train_production_model(
    dataset, eligibility, FilingMomentumModelConfig(),
    strategy_id="filing_momentum_ml", strategy_version="notebook",
)
print("blocked:", result.blocked)
print("reason:", result.blocked_reason)


## Findings

- In this environment, `blocked=True` -- scikit-learn is not installed, so no model was fit, and none was faked in its place.

## Limitations

- To exercise the real training path, install the `model` optional dependency group (`pip install -e '.[model]'`, scikit-learn>=1.3.0,<2.0.0) and re-run this cell -- do **not** install it silently as part of running this notebook.